# Web Map with Folium

**Folium** is a Python library for creating interactive web maps based on the JavaScript library Leaflet.js.

We have already used Folium many times — whenever we called the `.explore()` method on a `GeoDataFrame`. Under the hood, `GeoPandas` uses Folium to render those interactive maps.

Now we'll move to more flexible, hands-on map configuration directly through the Folium library.

In this section we will build an interactive map with the following layers:

- district boundary;
- land use;
- U-Bahn stations;
- walking isochrones (5, 10, and 15 minutes) from the stations.

## 0. Importing Libraries and Preparing Data

### 0.1. Importing Libraries

In [ ]:
import geopandas as gpd

import folium
from folium.plugins import MousePosition, Fullscreen, MiniMap

### 0.2. Preparing Data

In this example we use four datasets prepared in advance for Leopoldstadt — the second district of Vienna, an island between the Danube and the Danube Canal — in `data/leopoldstadt/`:

- `area.geojson` — district boundary;
- `landuse.geojson` — land use polygons;
- `metro.geojson` — U-Bahn stations;
- `isochrones.geojson` — walking isochrones from the stations.

_Every dataset and its source is listed on the [Course Modules](../module_0/syllabus.md) page._

#### 0.2.1. District Boundary

Read the data.

In [ ]:
area = gpd.read_file("../../data/leopoldstadt/area.geojson")

Inspect the data structure.

In [ ]:
area.head()

Preview the layer on a map.

In [ ]:
area.explore(tiles="cartodbpositron")

#### 0.2.2. Land Use

The land use polygons:

In [ ]:
landuse = gpd.read_file("../../data/leopoldstadt/landuse.geojson")

Its attribute table:

In [ ]:
landuse.head()

And on the map:

In [ ]:
landuse.explore(tiles="cartodbpositron")

#### 0.2.3. U-Bahn Stations

The U-Bahn stations:

In [ ]:
metro = gpd.read_file("../../data/leopoldstadt/metro.geojson")

Their attributes:

In [ ]:
metro.head()

On the map:

In [ ]:
metro.explore(tiles="cartodbpositron")

#### 0.2.4. Isochrones

And the isochrones:

In [ ]:
isochrones = gpd.read_file("../../data/leopoldstadt/isochrones.geojson")

Their attributes:

In [ ]:
isochrones.head()

On the map:

In [ ]:
isochrones.explore(tiles="cartodbpositron")

## 1. Creating the Base Map

Before adding thematic layers — the district boundary, land use, U-Bahn stations, and isochrones — we need to create the **map base**.

In this step we will:

1. determine the point where the map should open;
2. set the initial zoom level;
3. choose the background tile layer;
4. create the Folium map object that layers will be added to.

### 1.1. Setting the Map Centre

For the map to open centred on the study area, we calculate the geometric centroid of the district.

As we saw in the [second module](../module_2/projections_3.ipynb), geometric operations belong in a projected CRS, so we reproject the layer to its UTM zone, take the centroid there, and convert the result back to degrees — Folium expects latitude and longitude.

In [ ]:
centre = (
    area.to_crs(area.estimate_utm_crs())   # metric CRS
        .geometry.centroid                 # centroid of the district
        .to_crs(area.crs)                  # back to degrees
        .iloc[0]
)

centre_lat = centre.y
centre_lon = centre.x

print(f"Map centre: {centre_lat}, {centre_lon}")

### 1.2. Creating the Map Object

Now let's create the base Folium map:

- `folium.Map()` creates an interactive web map;
- `location` sets the map centre;
- `zoom_start` sets the initial zoom level — the larger the number, the closer the view; 13 frames a district like ours, while a whole city fits at 11–12;
- `tiles` selects the background tile layer;
- `control_scale` adds a scale bar to the map.

In [ ]:
m = folium.Map(
    location=[centre_lat, centre_lon],
    zoom_start=13,
    tiles="cartodbpositron",
    control_scale=True
)

m

## 2. Adding Layers

The base map is ready, but so far it contains only the background tile layer.

Now we'll gradually add spatial data as individual layers.

In Folium, data is typically added as `GeoJson` objects.

The general pattern looks like this:

```python
folium.GeoJson(
    data,
    display_parameters
).add_to(m)
```

### 2.1. District Boundary

We start with the most basic layer — the district boundary — to mark the study area.

#### 2.1.1. Defining the Style

The `style_function` is called separately for each GeoJSON feature. The `feature` object contains the geometry and attributes of that feature in `feature["properties"]`.

The style function must return a dictionary with the rendering parameters:

- `fillColor` — polygon fill colour;
- `color` — border colour;
- `weight` — line width;
- `fillOpacity` — fill opacity;
- `opacity` — border opacity;
- `dashArray` — makes the line dashed.

In [ ]:
def area_style(feature):
    return {
        "fillColor": "#64748B",
        "color": "#334155",
        "weight": 2,
        "fillOpacity": 0.03,
        "opacity": 0.65,
        "dashArray": "6, 5",
    }

#### 2.1.2. Adding the GeoJSON Layer

Add the layer to the map.

In [ ]:
folium.GeoJson(
    area,
    name="District Boundary",
    style_function=area_style,
).add_to(m)

m

### 2.2. Land Use

Now we move on to a more complex thematic layer — land use. Unlike the previous examples, this dataset contains multiple categories of features with different land use types. Before styling, we'll first look at which categories exist in the dataset, and if needed, merge some of them into broader groups.

This is the one layer here that does not come from OpenStreetMap. Vienna surveys the actual use of every parcel in the city and publishes the result as the **Realnutzungskartierung**, which covers the district wall to wall — no gaps — and classifies each polygon at three levels of detail: `LEV1` is the coarsest (three classes), `LEV2` has eleven, and `LEV3` goes down to individual uses such as a swimming pool or a railway yard. OpenStreetMap does have a `landuse` tag, but in Vienna it is mapped only patchily: it covers about two thirds of this district, and seven polygons in eight are a single lawn or flowerbed.

We will map `LEV2` — detailed enough to be interesting, coarse enough to fit in a legend.

#### 2.2.1. Exploring Categories

First, which land use types are present in the data.

In [ ]:
landuse["LEV2"].value_counts()

#### 2.2.2. Grouping Categories

Eleven categories is more than a reader can hold at once, so we'll merge related ones into seven broader groups. We create a mapping dictionary from the original land use types to the new groups.

The keys are the German category names exactly as they appear in the data. Leaving them untranslated is deliberate: these are the values the file actually contains, and a dictionary that quietly renames them would stop matching the moment you re-download the layer. The translation belongs in the legend, which we build further down.

In [ ]:
landuse_groups = {
    "Wohn- u. Mischnutzung (Schwerpunkt Wohnen)": "residential",

    "Geschäfts,- Kern- und Mischnutzung (Schwerpunkt betriebl. Tätigkeit)": "business",
    "Industrie- und Gewerbenutzung": "business",

    "soziale Infrastruktur": "social",

    "Erholungs- u. Freizeiteinrichtungen": "recreation",

    "Naturraum": "nature",
    "Landwirtschaft": "nature",

    "Gewässer": "water",

    "Straßenraum": "transport",
    "weitere verkehrliche Nutzungen": "transport",
    "Technische Infrastruktur/Kunstbauten/Sondernutzung": "transport",
}

#### 2.2.3. Colors for Each Group

Assign a distinct colour to each group.

Two of the seven are not free choices: water is blue and nature is green, because a map that breaks those conventions is read wrong before it is read at all. Transport takes a neutral grey — it is the background against which the rest is read, not a category anyone is looking for. The remaining four take distinct hues.

One caveat worth stating plainly: seven categorical fills is past the point where colour alone can be told apart reliably, particularly for readers with colour vision deficiency. That is why the legend below is not optional and the tooltip names the category — the colour narrows it down, the label settles it.

In [ ]:
landuse_colors = {
    "residential": "#eda100",
    "business": "#eb6834",
    "social": "#e87ba4",
    "recreation": "#1baf7a",
    "nature": "#008300",
    "water": "#2a78d6",
    "transport": "#9aa0a6",
}

#### 2.2.4. Style Function

Now let's write a function that automatically assigns a style to each feature.

`landuse_style(feature)` is called separately for each GeoJSON feature.
The `feature` argument is the current layer feature including its geometry and attributes, stored in `feature["properties"]`.

Inside the function:

1. The `LEV2` field value is extracted from the feature's attributes.
2. The `landuse_groups` dictionary maps the raw land use type to its broader group.
3. The group's colour is looked up in `landuse_colors`.
4. The function returns a dictionary with the rendering style for that feature.

Both lookups have a fallback: a land use type that is missing from `landuse_groups` falls into `"other"` and is drawn in neutral grey. All eleven categories the city uses are mapped, so nothing should reach the fallback — but it keeps the map working if a later survey adds one.

In [ ]:
def landuse_style(feature):

    landuse_type = feature["properties"].get("LEV2")

    landuse_class = landuse_groups.get(landuse_type, "other")

    color = landuse_colors.get(landuse_class, "#d9d9d9")

    return {
        "fillColor": color,
        "color": "#ffffff",
        "weight": 0.4,
        "fillOpacity": 0.5,
        "opacity": 0.6,
    }

#### 2.2.5. Configuring Tooltips

Tooltips appear when the user hovers over a feature.

The tooltip will display two fields: `LEV2`, the category we coloured the map by, and `LEV3`, the finer classification underneath it — so hovering over a green polygon tells the reader whether it is a park, a meadow or a wood.
`aliases` sets the field labels, and `localize=True` ensures values are formatted correctly.

In [ ]:
landuse_tooltip = folium.GeoJsonTooltip(
    fields=["LEV2", "LEV3"],
    aliases=["Land Use Type:", "Detail:"],
    localize=True
)

#### 2.2.6. Adding the Layer to the Map

Now we add the land use layer to the map using `folium.GeoJson`, passing the data, the style function, and the tooltips.

`style_function=landuse_style` means Folium will call the style function for each GeoJSON feature, automatically determining its colour and rendering parameters based on its attributes.

In [ ]:
folium.GeoJson(
    landuse,
    name="Land Use",
    style_function=landuse_style,
    tooltip=landuse_tooltip
).add_to(m)

m

### 2.3. Walking Isochrones

Isochrones delineate the area reachable within a given travel time. In our example we use walking isochrones for three time thresholds: 5, 10, and 15 minutes.

#### 2.3.1. Defining Styles for Each Zone

First, let's create a dictionary of rendering styles for each isochrone.
The keys are travel times in seconds:

- `300` — 5 minutes;
- `600` — 10 minutes;
- `900` — 15 minutes.

For each zone we set the line rendering parameters:

- `color` — line colour;
- `weight` — line width;
- `opacity` — opacity.

In [ ]:
isochrone_styles = {
    300: {
        "color": "#475569",
        "weight": 3.5,
        "opacity": 0.95,
    },
    600: {
        "color": "#64748B",
        "weight": 3,
        "opacity": 0.85,
    },
    900: {
        "color": "#94A3B8",
        "weight": 2.5,
        "opacity": 0.75,
    },
}


#### 2.3.2. Style Function

Now let's write a function that automatically assigns a style to each isochrone.

As in the previous examples, the function receives a `feature` object — a single GeoJSON feature with its attributes. Here, the attributes store the travel time value.

The travel time in seconds is extracted from `feature["properties"]["value"]`:

- `300` — 5 minutes;
- `600` — 10 minutes;
- `900` — 15 minutes.

The rendering parameters for that zone are then looked up in `isochrone_styles`. If the value is not in the dictionary, a default style is used.

The function returns a dictionary of line rendering parameters for the isochrone.

In [ ]:
def isochrone_style(feature):
    value = int(feature["properties"].get("value", 0))
    style = isochrone_styles.get(value, {
        "color": "#9CA3AF",
        "weight": 2,
        "opacity": 0.7,
    })

    return {
        "fill": False,
        "color": style["color"],
        "weight": style["weight"],
        "opacity": style["opacity"],
    }

#### 2.3.3. Preparing the Data

Before adding the layer, let's prepare the dataset for display.

We create a separate GeoDataFrame `isochrones_layer` containing only the necessary fields:

- `station_name` — the station name;
- `value` — travel time in seconds;
- `geometry` — the isochrone geometry.

We then cast `value` to integer and add a new `minutes` field storing travel time in minutes — handy for tooltips and labels.

In [ ]:
isochrones_layer = isochrones[["station_name", "value", "geometry"]].copy()

isochrones_layer["value"] = isochrones_layer["value"].astype(int)
isochrones_layer["minutes"] = (isochrones_layer["value"] / 60).astype(int)

#### 2.3.4. Configuring Tooltips

The isochrones get tooltips of their own.

The tooltip will display:

- `station_name` — the station name;
- `minutes` — travel time in minutes.

`aliases` sets the field labels, and `sticky=True` keeps the tooltip pinned near the cursor.

In [ ]:
isochrones_tooltip = folium.GeoJsonTooltip(
    fields=["station_name", "minutes"],
    aliases=["Station:", "Walking minutes:"],
    sticky=True
)

#### 2.3.5. Adding the Layer to the Map

Add the isochrone layer to the map using `folium.GeoJson`.

`style_function=isochrone_style` tells Folium to call `isochrone_style` for each feature, selecting the line colour, width, and opacity based on the travel time value.

In [ ]:
folium.GeoJson(
    isochrones_layer,
    name="Isochrones",
    style_function=isochrone_style,
    tooltip=isochrones_tooltip
).add_to(m)

m

### 2.4. U-Bahn Stations

The last layer — U-Bahn stations.

Unlike the previous examples, here we work with point features. Points are typically displayed using markers.

#### 2.4.1. Styling the Points

We'll use `folium.CircleMarker` — circular markers with configurable size and style.

Parameters used here:

- `radius` — marker size;
- `color` — border colour;
- `weight` — border width;
- `fill` — enables fill;
- `fill_color` — fill colour;
- `fill_opacity` — fill opacity.

In [ ]:
metro_markers = folium.CircleMarker(
    radius=6,
    color="#FFFFFF",
    weight=2,
    fill=True,
    fill_color="#334155",
    fill_opacity=1,
)

#### 2.4.2. Configuring Tooltips

And tooltips for the stations.

The tooltip displays the `name` field with the station name.
`aliases` sets the field label, and `sticky=True` pins the tooltip near the cursor.

In [ ]:
metro_tooltip = folium.GeoJsonTooltip(
    fields=["name"],
    aliases=["Station:"],
    sticky=True
)

#### 2.4.3. Adding the Layer to the Map

Now we add the U-Bahn stations layer using `folium.GeoJson`:

- `metro` — the GeoDataFrame with station features;
- `marker=metro_markers` — the circle marker style defined above;
- `tooltip=metro_tooltip` — tooltips with station names.

In [ ]:
folium.GeoJson(
    metro,
    name="U-Bahn Stations",
    marker=metro_markers,
    tooltip=metro_tooltip,
).add_to(m)

m

## 3. Map Controls

After adding all thematic layers, let's configure the map controls.

### 3.1. Layer Control

Add a layer control panel that lets users toggle individual layers on and off.

In [ ]:
folium.LayerControl(collapsed=False).add_to(m)

`LayerControl` lets users toggle layers on and off. `collapsed=False` keeps the panel expanded by default.

### 3.2. Cursor Coordinates

Add a cursor coordinate display. When the user moves the mouse over the map, the current latitude and longitude are shown.

In [ ]:
MousePosition().add_to(m)

### 3.3. Fullscreen Button

Add a button that expands the map to the full browser window — useful when the map is embedded in a page alongside other content.

Parameters:

- `position="bottomright"` places the button in the bottom-right corner;
- `title` sets the tooltip text when entering fullscreen;
- `title_cancel` sets the tooltip text when exiting fullscreen;
- `force_separate_button=True` renders the button as a standalone UI element.

In [ ]:
Fullscreen(
    position="bottomright",
    title="Enter fullscreen",
    title_cancel="Exit fullscreen",
    force_separate_button=True,
).add_to(m)

### 3.4. Mini Map

The mini map helps users understand where the current map view sits relative to a broader area.

In [ ]:
MiniMap(tile_layer="cartodbpositron", toggle_display=True).add_to(m)

### 3.5. Legend

A map with seven land use colours and three isochrone line weights is unreadable without a legend — the reader has no way to tell what a colour means.

Folium has no built-in legend for `GeoJson` layers. The `branca` colour maps that Folium ships with draw a **continuous gradient bar**, which suits a numeric scale (population, density, year of construction) but misrepresents categories: land use groups have no order, and a gradient implies one. So for a categorical layer the usual approach is to add a small block of HTML on top of the map.

Two decisions are worth spelling out:

1. **We generate the legend from the same dictionaries that drive the styling** — `landuse_colors` and `isochrone_styles`. This is the whole point: a legend written by hand drifts out of sync as soon as someone changes a colour, and a map whose legend lies is worse than a map with no legend at all. Here, changing a colour in `landuse_colors` changes both the map and its legend.
2. **The legend is attached to the map root, not added as a layer** — via `m.get_root().html.add_child(...)`. It is page furniture rather than spatial data, so it should not appear in the layer control panel and should not disappear when a layer is switched off.

The only thing we still need is a set of human-readable labels. This is where the German category names get translated — and where the short group keys (`nature`, `transport`), fine in code, turn into something a reader can use.

In [ ]:
landuse_labels = {
    "residential": "Housing and mixed use",
    "business": "Business and industry",
    "social": "Social infrastructure",
    "recreation": "Recreation and leisure",
    "nature": "Nature and farmland",
    "water": "Water",
    "transport": "Transport and utilities",
}

Now we assemble the legend itself. The two loops build one row per land use group and one row per isochrone zone; the swatch of each row is painted with the colour that the style function will use, and each isochrone line is drawn with the same width and colour as on the map.

The block is styled with a little CSS: `position: fixed` pins it to the corner of the map frame, `z-index` keeps it above the tiles, and the bottom offset leaves room for the scale bar.

One detail to keep in mind: on the map the polygons are drawn with `fillOpacity` 0.5, so the basemap shows through and the colours look paler there than in the legend swatches. Full-strength swatches stay legible at 14 pixels, which is why they are drawn that way here.

In [ ]:
# Land use: one row per group, the swatch colour taken from landuse_colors
landuse_rows = "".join(
    f"""
    <div class="legend-row">
        <span class="legend-swatch" style="background: {landuse_colors[group]}"></span>
        {label}
    </div>"""
    for group, label in landuse_labels.items()
)

# Isochrones: one row per zone, the line styled exactly as on the map
isochrone_rows = "".join(
    f"""
    <div class="legend-row">
        <span class="legend-line" style="border-top: {style["weight"]}px solid {style["color"]}"></span>
        {seconds // 60} min walk
    </div>"""
    for seconds, style in isochrone_styles.items()
)

legend_html = f"""
<div class="map-legend">
    <div class="legend-title">Land use</div>
    {landuse_rows}
    <div class="legend-title">Walking isochrones</div>
    {isochrone_rows}
</div>

<style>
.map-legend {{
    position: fixed;
    bottom: 42px;   /* leaves room for the scale bar */
    left: 12px;
    z-index: 9999;
    background: rgba(255, 255, 255, 0.92);
    padding: 10px 14px;
    border: 1px solid #cbd5e1;
    border-radius: 6px;
    font-family: sans-serif;
    font-size: 12px;
    line-height: 1.6;
    color: #1e293b;
}}
.legend-title {{ font-weight: 600; margin: 4px 0 2px; }}
.legend-row {{ display: flex; align-items: center; gap: 8px; }}
.legend-swatch {{
    width: 14px;
    height: 14px;
    border: 1px solid #ffffff;
    border-radius: 2px;
    display: inline-block;
}}
.legend-line {{ width: 18px; display: inline-block; }}
</style>
"""

m.get_root().html.add_child(folium.Element(legend_html))

m

## 4. Viewing and Saving the Map

The map is now complete.

We have added:

- thematic layers;
- feature styling;
- tooltips;
- map controls.

Time to look at the finished map and save it as an HTML file.

### 4.1. Final Map

Display the finished interactive map.

In [ ]:
m

### 4.2. Saving the Map

Save the map to an HTML file — it can then be opened in any browser. The path is relative to the notebook, so the file lands next to it. The line is left commented out here; uncomment it when you want the file, and see the [next section](map_2.ipynb) for publishing it online.

In [ ]:
# m.save("index.html")

## Summary

In this section we built an interactive web map with multiple layers.

We:

- loaded and inspected the data;
- created a base map;
- added thematic layers;
- configured styles and tooltips;
- added map controls;
- looked at how to save the map as HTML — publishing it is the subject of the [next section](map_2.ipynb).